# A0. Custom mechanisms

In this tutorial, we dive into the nitty gritty of how membrane mechanisms are implemented in AxonML. We will explore how to define custom ion channels / dynamics and synaptic mechanisms, and how to integrate them into your neuronal models.



## Mechanism vs State

AxonML separates the *Mechanism* (user-facing membrane mechanism) from the *State* helper used to declare per-compartment state variables and their dynamics.

- **Mechanism** (`Mechanism`): declares parameters, states, ionic usage, currents, and lifecycle hooks.
- **State** (`mechanisms.State`): inner helper class where you declare state variables and their ODEs/kinetics; register with `Mechanism.STATE(...)`.


## Declaring state variables and dynamics

Use a `State` subclass to name state variables and define their dynamics with `DERIVATIVE` or `KINETIC`. `breakpoint` computes `ASSIGNED` variables (steady states, time constants, etc.) that appear in derivatives/currents but are not themselves state or parameter variables.


In [ ]:
import torch
from axonml.models.mechanisms import Mechanism as M
from axonml.models.mechanisms import State as S
from axonml.models.mechanisms.ops import exp, vtrap

class mhn(S):
    method = "cnexp"
    has_q10 = True  # optional Q10 scaling hook

    S.STATE("m", "h", "n")
    S.DERIVATIVE(
        "m' = (minf - m) / mtau",
        "h' = (hinf - h) / htau",
        "n' = (ninf - n) / ntau",
    )
    S.ASSIGNED("minf", "mtau", "hinf", "htau", "ninf", "ntau")

    def calc_q10(self):
        return 3.0 ** ((self.celsius - 6.3) / 10.0)

    def breakpoint(self, v, states=None):
        q10 = self.q10()
        alpha_m = 0.1 * vtrap(-(v + 40), 10)
        beta_m = 4 * exp(-(v + 65) / 18)
        tot = alpha_m + beta_m
        mtau = 1 / (q10 * tot)
        minf = alpha_m / tot

        alpha_h = 0.07 * exp(-(v + 65) / 20)
        beta_h = 1 / (exp(-(v + 35) / 10) + 1)
        tot = alpha_h + beta_h
        htau = 1 / (q10 * tot)
        hinf = alpha_h / tot

        alpha_n = 0.01 * vtrap(-(v + 55), 10)
        beta_n = 0.125 * exp(-(v + 65) / 80)
        tot = alpha_n + beta_n
        ntau = 1 / (q10 * tot)
        ninf = alpha_n / tot

        # return dict whose keys match ASSIGNED names
        return {"mtau": mtau, "minf": minf, "htau": htau, "hinf": hinf, "ntau": ntau, "ninf": ninf}

    def inf(self, v):
        states = self.breakpoint(v, None)
        return {"m": states["minf"], "h": states["hinf"], "n": states["ninf"]}


class hh(M):
    """
    Classic Hodgkin-Huxley mechanism with m/h/n gates and leak.
    """

    M.STATE(mhn)
    M.GLOBAL(gnabar=0.12, gkbar=0.036, gl=0.0003, ena=50.0, ek=-77.0, el=-54.3)
    M.NONSPECIFIC_CURRENT("il", "ina", "ik")
    # Example for ions: M.USEION("na", read=("nai", "nao", "ena"), write=("ina",))

    def il(self, v):
        return self.gl * (v - self.el)

    def ina(self, v):
        gna = self.gnabar * self.m**3 * self.h
        return gna * (v - self.ena)

    def ik(self, v):
        gk = self.gkbar * self.n**4
        return gk * (v - self.ek)


### Methods of integration

AxonML supports multiple methods of integrating state variable dynamics, specified by the `method` class attribute of the `State` subclass. Supported methods are:

- `"cnexp"`: exact integration for linear ODEs e.g., `dx/dt = (x_inf - x) / tau_x`.
    - This is the default method, is fast and numerically stable, and should be preferred when applicable. From a correctness standpoint, it is equivalent to using an exponential Euler method with timestep equal to the simulation timestep. This may also be used for coupled linear ODEs that can be solved analytically; in that case, states that appear in the RHS of other derivatives are treated as constants during integration (assuming their value at the beginning of the timestep). This is not strictly exact, but is often a perfectly good approximation (e.g., see `caintscale`) and is generally much faster than a full coupled solve.
- `"derivimplicit"`: implicit integration for general ODEs using Newton-Raphson method.
    - This is a more general method that can be used for nonlinear and coupled ODEs. It is more computationally expensive than `cnexp` and may have convergence issues for stiff systems or poor initial guesses, but is more widely applicable.
    - `derivimplicit` constructs and solves a system of equations representing the implicit update of all state variables simultaneously. It will attempt to compute analytical Jacobians for better performance and convergence; if this fails, it will fall back to numerical Jacobians which can reduce performance.

To specify the integration method, set the `method` attribute in your `State` subclass. For example, to use the `derivimplicit` method:
```python
class MyMechanismState(State):
    method = "derivimplicit"
    # Define states and dynamics here
```


### Q10 scaling
- Set `has_q10 = True` on a `State` subclass to enable temperature scaling. AxonML then exposes `self.q10()` in the state, calling `calc_q10()` if provided (otherwise defaulting to 1).
- Implement `calc_q10(self)` to return the multiplicative factor as a function of `self.celsius`. In the HH gates above:
  ```python
  def calc_q10(self):
      return 3.0 ** ((self.celsius - 6.3) / 10.0)
  ```
- Use `self.q10()` inside `breakpoint` or kinetic rate calculations to scale time constants and rates.



### State `inf` helper
- Implement `inf(self, v)` on a `State` to return a mapping of steady-state values for the declared state variables (e.g., `{"m": minf, "h": hinf}`) at a given voltage. AxonML uses this during initialization to set state variables unless overridden.
- You can also use `inf` to encode any custom initialization rule you want; it’s invoked during `initialize()` for states.



### Multiple State bundles per Mechanism
A mechanism can register multiple independent `State` subclasses. For example, the HH gates `m`, `h`, and `n` could each be defined as separate `State` classes and then registered via multiple `M.STATE(...)` calls (or a single call with multiple states). AxonML will aggregate all declared states/derivatives and step them together.



### Buffers and `initial`
- `BUFFER(name, ...)` declares auxiliary per-compartment buffers stored on the state (not evolved by an ODE). They are useful for geometry- or configuration-dependent terms reused in derivatives.
- Implement `initial(self, v)` on a `State` to populate these buffers (and optionally states) at initialization. It runs once during `initialize()` and can depend on morphology (e.g., `self.diam`).
- Example (extracellular Ca accumulation in `caextscale`):
  ```python
  class cao(S):
      S.STATE("cao")
      S.GLOBAL(lseg=1.0, txfer=4511.0, FARADAY=96500.0, cabath=2.0, fhspace=1.0)
      S.BUFFER("SA", "Vol_peri")
      S.DERIVATIVE("cao' = (ica*SA/(2*Vol_peri*FARADAY) + (cabath - cao)/txfer)")

      def initial(self, v):
          self.SA = math.pi * (1e-4) * self.diam * self.lseg
          Vol = math.pi * ((1e-4) * (self.diam / 2)) ** 2 * self.lseg
          self.Vol_peri = (
              math.pi * ((1e-4) * ((self.diam + self.fhspace) / 2)) ** 2 * self.lseg
          ) - Vol
  ```
  `SA` and `Vol_peri` are computed once, then reused inside the derivative during simulation.


## API overview
### State classmethods (at `State` definition)
- `STATE(name, ...)`: declare state variables.
- `DERIVATIVE("x' = expr", ...)`: symbolic ODEs for states.
- `KINETIC("~ a <-> b (alpha, beta)", ...)`: Markov/kinetic schemes between states.
- `ASSIGNED(name, ...)`: computed per-compartment variables; set in `breakpoint`.
- `GLOBAL` / `RANGE`: shared vs per-compartment parameters scoped to the State.
- `BUFFER(name, ...)`: declare auxiliary per-compartment buffers initialized in `initial` and reused in derivatives/assigned calculations.


### State lifecycle hooks
- `initial(self, v)`: optional initializer run during `initialize()` to set buffers/states (e.g., geometry-dependent buffers declared via `BUFFER`).
- `breakpoint(self, v, states=None)`: compute `ASSIGNED`/intermediate values each step before derivatives/currents are evaluated; return a dict matching `ASSIGNED` names if you choose to return values.
- `inf(self, v)`: return steady-state values for state variables (used for initialization unless overridden by the caller).
- `calc_q10(self)`: optional helper when `has_q10=True`; returns the temperature scaling factor consumed by `self.q10()` inside your kinetics.


### Mechanism classmethods (at mechanism definition)
- `STATE(StateSubclass, ...)`: register State bundles used by the mechanism.
- `GLOBAL`, `RANGE`: shared vs per-compartment parameters on the mechanism.
- `ASSIGNED(name, ...)`: per-compartment buffers/computed vars (analogous to `BUFFER` in State); typically set in `initial`/`breakpoint`.
- `INIT(...)`: initialization hooks for mechanism buffers/states.
- `USEION(ion, read=(), write=())`: declare ionic dependencies. `read`/`write` must be in `{ion}i`, `{ion}o`, `e{ion}`, `i{ion}` (e.g., read `("ena", "nai", "nao")`, write `("ina",)` to contribute Na current).
- `NONSPECIFIC_CURRENT(name, ...)`: register membrane currents not tied to an ion.
- `EXPLICIT(name, ...)`: mark voltage-independent currents when assembling conductance terms.



### Mechanism lifecycle hooks
- `initial(self, v)`: optional initializer run during `initialize()`; set mechanism-level `ASSIGNED` buffers or other cached quantities.
- `breakpoint(self, v)`: compute mechanism currents and mechanism-level `ASSIGNED` values each step (called before current assembly).
- Current methods (`ina`, `ik`, `il`, etc.): implement currents declared via `NONSPECIFIC_CURRENT` or ion writes in `USEION`. These are invoked to assemble total membrane/ionic currents.



**Example (h-channel with mechanism-level ASSIGNED):**
```python
class h(M):
    M.STATE(s, f)
    M.GLOBAL(gbar=0.0001)
    M.USEION("k", read=["ek"], write=["ik"])
    M.USEION("na", read=["ena"], write=["ina"])
    M.ASSIGNED("g")  # mechanism-level buffer, set in initial/breakpoint

    def initial(self, v):
        self.breakpoint(v)  # compute g at t=0

    def breakpoint(self, v):
        self.g = self.gbar * (0.5 * self.s + 0.5 * self.f)

    def ina(self, v):
        return 0.5 * self.g * (v - self.ena)

    def ik(self, v):
        return 0.5 * self.g * (v - self.ek)
```
Here `ASSIGNED("g")` acts like a mechanism-level buffer (analogous to `BUFFER` in State) that is populated in `initial`/`breakpoint` and reused in current computations.
